# Выгрузка базового контекста кейсов для агента

Notebook собирает читаемый базовый набор данных по CSI-ответам и антифрод-сработкам, а затем формирует строковые запросы для агента. Логика разделена на конфигурацию, функции загрузки, функции объединения и форматирование, чтобы новые колонки и источники добавлялись через отдельные списки/спеки.

Важно: notebook не использует API-ключи. Он ожидает, что в окружении уже доступны `spark`, `eng_pg`, `pd`, `np` и `pyspark.sql.functions as F` либо они будут импортированы в первой ячейке.

In [ ]:
"""Выгрузка базового контекста антифрод-кейсов для передачи агенту.

Содержит:
- build_week_windows: рассчитывает недельные окна для исторической выгрузки.
- safe_to_spark: преобразует pandas DataFrame в Spark DataFrame с явной схемой.
- load_csi_answers: загружает CSI-ответы клиентов за период.
- transform_survey_data_clean: разворачивает вопросы CSI в широкий формат.
- load_hits_extra: загружает основную информацию по сработкам.
- coalesce_duplicate_columns: объединяет дублирующиеся колонки после join/merge.
- add_business_channel: добавляет бизнес-канал по sub_channel.
- load_atm_mcc: загружает MCC по event_id из истории автомаркировки.
- load_mcc_dictionary: загружает справочник MCC с описанием.
- enrich_mcc_names: применяет ручные бизнес-названия MCC.
- load_day_events: загружает события клиентов за день сработки.
- build_base_case_dataset: собирает итоговый базовый dataset.
- parse_complex_string: безопасно разбирает JSON-like строки.
- format_complex_value: форматирует сложные значения в читаемый текст.
- format_dataframe_rows_to_list: превращает строки DataFrame в список запросов агенту.
- build_agent_requests: формирует финальные текстовые запросы для агента.
"""

from __future__ import annotations

import ast
import json
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any

import numpy as np
import pandas as pd
import pyspark.sql.types as T
from pyspark.sql import DataFrame as SparkDataFrame
from pyspark.sql import functions as F

## 1. Конфигурация периода, таблиц и колонок

In [ ]:
@dataclass(frozen=True)
class ExportConfig:
    """Конфигурация выгрузки базового набора данных.

    Args:
        date_from: Начальная дата целевого периода в формате YYYY-MM-DD.
        date_to: Конечная дата целевого периода в формате YYYY-MM-DD.
        digest_week: Номера недель дайджеста, где последний элемент считается текущей неделей.
        history_weeks: Количество недель истории, которое нужно добавить к целевому периоду.
        csi_table: Таблица CSI-ответов клиентов.
        hits_extra_table: Таблица с расширенной информацией по сработкам.
        automarking_table: Таблица истории автомаркировки для MCC.
        cards_event_table: Таблица событий карт.
        uko_event_table: Таблица событий UKO.

    Returns:
        Объект с параметрами выгрузки, который передается функциям загрузки.
    """

    date_from: str = "2026-03-28"
    date_to: str = "2026-04-10"
    digest_week: list[int] = field(default_factory=lambda: [14, 15])
    history_weeks: int = 6
    csi_table: str = "csp_repo_features3.history_csi_clients_answers_v2_129372114_129377108"
    hits_extra_table: str = "cspfs_repo_features3.hits_extra_info_129372427_view"
    automarking_table: str = "csp_repo_features.history_automarking_big_148078_155487"
    cards_event_table: str = "csp_afpc_sss_inc.cards_event"
    uko_event_table: str = "csp_afpc_sss_inc.uko_event"


CONFIG = ExportConfig()

HITS_EXTRA_COLUMNS = [
    "epk_id", "event_id", "event_channel", "sub_channel", "event_type", "age",
    "segment", "age_category", "resolution_first", "resolution_first_dttm",
    "resolution_last", "resolution_last_dttm", "surface", "atm_merchant_name",
    "merchant_info", "source_type_accept", "hits_extra_facts", "client_balance",
    "recipient_info", "scoring_oss", "previous_events_additional_info",
    "posterious_events_additional_info", "previous_events", "posterious_events",
]

CHANNEL_BY_SUB_CHANNEL = {
    "UFS.MOBILEAPI": "ДБО",
    "ISSUER_ACQUIRER": "Эмиссия",
    "UFS.BRANCHAPI": "ВСП",
    "ISSUER": "Эмиссия",
    "UFS.WEBAPI": "ДБО",
    "ESA.MOBILEAPI": "ДБО",
    "ISSUER_WEBACQUIRER": "Эмиссия",
    "ATMAPI": "ДБО",
    "UFS.MBKAPI": "ДБО",
    "UFS.ATMAPI": "ДБО",
    "UFS.OTHER": "ДБО",
    "ESA.WEBAPI": "ДБО",
    "ESA.BRANCHAPI": "ВСП",
}

MCC_NAME_OVERRIDES = {
    "6009": "Микрофинансовые организации",
    "3990": "Экосистема Яндекса",
    "3991": "Экосистема Сбера",
    "9390": "Госуслуги",
    "5262": "Маркетплейсы",
    "9406": "Микрофинансовые организации",
}

EVENT_EXTRA_COLUMNS = [
    "main_rule", "rules", "subrules", "list_geo_1km_user_id", "device_source_sdk",
    "user_ip_location_country_code", "user_ip_location_city",
    "payee_communication_term_with_ak_pf", "virus_names", "event_id",
]

EVENT_STRUCT_COLUMNS = [
    "evt_time", "event_type", "sub_type", "event_channel", "event_description",
    "transaction_amount", "atm_mcc", *EVENT_EXTRA_COLUMNS,
]

## 2. Расчет окон дат

In [ ]:
def build_week_windows(config: ExportConfig) -> tuple[list[list[Any]], str, str, str]:
    """Рассчитывает недельные окна и технические даты для фильтров Spark.

    Args:
        config: Конфигурация выгрузки с целевым периодом и числом недель истории.

    Returns:
        Кортеж `(weeks_list, history_date_from, date_from_tst, date_to_tst)`, где
        `weeks_list` содержит окна `[start_week, end_week, week_num]`,
        `history_date_from` является началом исторического окна,
        `date_from_tst` и `date_to_tst` используются для таблиц с датой YYYYMMDD.
    """

    date_to_ts = pd.to_datetime(config.date_to)
    current_week_start = (date_to_ts - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
    weeks_list: list[list[Any]] = []

    for num in range(config.history_weeks, 0, -1):
        start_week = (pd.to_datetime(current_week_start) - pd.Timedelta(days=7 * num)).strftime("%Y-%m-%d")
        end_week = (date_to_ts - pd.Timedelta(days=7 * num)).strftime("%Y-%m-%d")
        week_num = config.digest_week[-1] - num
        weeks_list.append([start_week, end_week, week_num])

    weeks_list.append([current_week_start, config.date_to, config.digest_week[-1]])
    history_date_from = weeks_list[0][0]
    date_from_tst = pd.to_datetime(history_date_from).strftime("%Y%m%d")
    date_to_tst = pd.to_datetime(config.date_to).strftime("%Y%m%d")
    return weeks_list, history_date_from, date_from_tst, date_to_tst
\

weeks_list, history_date_from, date_from_tst, date_to_tst = build_week_windows(CONFIG)
print(CONFIG.date_from, CONFIG.date_to)
print(history_date_from, date_from_tst, date_to_tst)
weeks_list

## 3. Функции загрузки и нормализации базовых данных

In [ ]:
def safe_to_spark(pdf: pd.DataFrame) -> SparkDataFrame:
    """Преобразует pandas DataFrame в Spark DataFrame с предсказуемой схемой.

    Args:
        pdf: pandas DataFrame с базовыми кейсами.

    Returns:
        Spark DataFrame, где числовые, временные и строковые поля приведены к явным типам.
    """

    prepared = pdf.copy()
    object_columns = prepared.select_dtypes(include=["object"]).columns
    prepared[object_columns] = prepared[object_columns].astype(str).replace("nan", None)

    fields: list[T.StructField] = []
    for column_name, dtype in prepared.dtypes.items():
        dtype_text = str(dtype).lower()
        if column_name == "epk_id":
            fields.append(T.StructField(column_name, T.LongType(), True))
        elif "int" in dtype_text:
            fields.append(T.StructField(column_name, T.LongType(), True))
        elif "float" in dtype_text:
            fields.append(T.StructField(column_name, T.DoubleType(), True))
        elif "datetime" in dtype_text:
            fields.append(T.StructField(column_name, T.TimestampType(), True))
        else:
            fields.append(T.StructField(column_name, T.StringType(), True))

    return spark.createDataFrame(prepared, schema=T.StructType(fields))


def normalize_type_accept(column: Any) -> Any:
    """Возвращает Spark-выражение для нормализации канала подтверждения.

    Args:
        column: Spark Column с исходным значением `type_accept`.

    Returns:
        Spark Column с унифицированным названием канала подтверждения.
    """

    return (
        F.when(column == "ЕРКЦ", F.lit("ЕРКЦ"))
        .when(column == "HINT Эмиссия", F.lit("HINT Карты"))
        .when(column == "HINT ДБО", F.lit("HINT ДБО"))
        .when(column == "Сотрудник ВСП", F.lit("Сотрудник ВСП"))
        .when(column == "ГПМ", F.lit("ГПМ"))
        .when(column == "IVR", F.lit("IVR"))
        .when(column == "ChipPin", F.lit("Chip+Pin"))
        .when(column == "РО/ЗРО", F.lit("РО/ЗРО"))
        .when(column == "Black List", F.lit("Black List (БА)"))
        .when(column == "Сотрудник Бета", F.lit("Сотрудник Бета"))
        .when(column == "БИО", F.lit("BIO"))
        .when(column == "ДБ", F.lit("ДБ"))
        .when(column == "Подтверждение на близкого", F.lit("Подтверждение на близкого"))
        .when(column == "Сотрудник Гамма", F.lit("Сотрудник Гамма"))
        .otherwise(F.lit("Не определено/None"))
    )


def load_csi_answers(config: ExportConfig, history_date_from: str) -> SparkDataFrame:
    """Загружает CSI-ответы клиентов и добавляет нормализованный канал подтверждения.

    Args:
        config: Конфигурация выгрузки.
        history_date_from: Начало исторического окна в формате YYYY-MM-DD.

    Returns:
        Spark DataFrame с CSI-ответами за расширенный период.
    """

    return (
        spark.table(config.csi_table)
        .filter(F.col("answer_date").between(history_date_from, config.date_to))
        .withColumn("type_accept_corrected", normalize_type_accept(F.col("type_accept")))
    )


def transform_survey_data_clean(
    df: pd.DataFrame,
    group_columns: list[str] | None = None,
    question_count: int = 8,
) -> pd.DataFrame:
    """Разворачивает строки вопросов CSI в широкий формат без дублирования колонок.

    Args:
        df: pandas DataFrame с CSI-ответами, где вопросы лежат в строках.
        group_columns: Колонки, задающие один кейс. По умолчанию используются доступные `epk_id`, `event_id`, `answer_date`.
        question_count: Максимальное количество вопросов, которое нужно развернуть в колонки.

    Returns:
        pandas DataFrame, где каждый кейс представлен одной строкой с колонками `question_N`, `answer_N`, `comment_N`.
    """

    if group_columns is None:
        group_columns = [column for column in ["epk_id", "event_id", "answer_date"] if column in df.columns]
    if not group_columns:
        raise ValueError("Нужна хотя бы одна колонка группировки для разворота CSI-ответов.")

    result_rows: list[dict[str, Any]] = []
    question_columns = {"question_number", "question_text", "answer_text", "comment_text"}

    for _, group in df.groupby(group_columns, dropna=False):
        base_row: dict[str, Any] = {}
        for column in df.columns:
            if column in question_columns:
                continue
            non_empty_values = group[column].dropna()
            base_row[column] = non_empty_values.iloc[0] if not non_empty_values.empty else None

        questions: dict[int, dict[str, Any]] = {}
        for _, row in group.iterrows():
            question_number = row.get("question_number")
            if pd.isna(question_number):
                continue
            try:
                question_number_int = int(str(question_number).strip())
            except (ValueError, TypeError):
                continue
            if 1 <= question_number_int <= question_count:
                questions[question_number_int] = {
                    "question": row.get("question_text"),
                    "answer": row.get("answer_text"),
                    "comment": row.get("comment_text"),
                }

        for question_number in range(1, question_count + 1):
            question_data = questions.get(question_number, {})
            base_row[f"question_{question_number}"] = question_data.get("question")
            base_row[f"answer_{question_number}"] = question_data.get("answer")
            base_row[f"comment_{question_number}"] = question_data.get("comment")

        result_rows.append(base_row)

    return pd.DataFrame(result_rows)

In [ ]:
def load_hits_extra(config: ExportConfig, event_ids: list[str], date_from_tst: str, date_to_tst: str) -> pd.DataFrame:
    """Загружает расширенную информацию по сработкам из hits_extra.

    Args:
        config: Конфигурация выгрузки.
        event_ids: Список event_id из CSI-ответов.
        date_from_tst: Начальная дата фильтра в формате YYYYMMDD.
        date_to_tst: Конечная дата фильтра в формате YYYYMMDD.

    Returns:
        pandas DataFrame с основными полями сработки.
    """

    if not event_ids:
        return pd.DataFrame(columns=HITS_EXTRA_COLUMNS)

    hits_extra = (
        spark.read.table(config.hits_extra_table)
        .filter(F.col("event_dt").between(date_from_tst, date_to_tst))
        .filter(F.col("event_id").isin(event_ids))
        .select(*HITS_EXTRA_COLUMNS)
    )
    return hits_extra.toPandas()


def coalesce_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Объединяет дублирующиеся колонки, выбирая первое непустое значение слева направо.

    Args:
        df: pandas DataFrame после merge, где могли появиться одинаковые имена колонок.

    Returns:
        pandas DataFrame без дублей колонок.
    """

    normalized = df.copy()
    normalized.columns = normalized.columns.str.replace("_x$", "", regex=True).str.replace("_y$", "", regex=True)
    normalized = normalized.replace("", np.nan)

    result: dict[str, pd.Series] = {}
    for column_name in normalized.columns.unique():
        duplicate_columns = [normalized.iloc[:, index] for index, column in enumerate(normalized.columns) if column == column_name]
        merged = duplicate_columns[0]
        for duplicate_column in duplicate_columns[1:]:
            merged = merged.combine_first(duplicate_column)
        result[column_name] = merged
    return pd.DataFrame(result)


def add_business_channel(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет бизнес-канал сработки на основе `sub_channel`.

    Args:
        df: pandas DataFrame с колонкой `sub_channel`.

    Returns:
        pandas DataFrame с колонкой `channel`.
    """

    result = df.copy()
    result["channel"] = result.get("sub_channel", pd.Series(index=result.index)).map(CHANNEL_BY_SUB_CHANNEL).fillna("Не определено")
    return result


def load_atm_mcc(config: ExportConfig, event_ids: list[str], date_from_tst: str, date_to_tst: str) -> pd.DataFrame:
    """Загружает MCC-код по event_id из истории автомаркировки.

    Args:
        config: Конфигурация выгрузки.
        event_ids: Список event_id для фильтрации.
        date_from_tst: Начальная дата фильтра в формате YYYYMMDD.
        date_to_tst: Конечная дата фильтра в формате YYYYMMDD.

    Returns:
        pandas DataFrame с колонками `event_id` и `mcc_code`.
    """

    if not event_ids:
        return pd.DataFrame(columns=["event_id", "mcc_code"])

    atm_merchant = (
        spark.table(config.automarking_table)
        .withColumn("event_dt", F.date_format(F.col("event_time"), "yyyyMMdd"))
        .filter(F.col("event_dt").between(date_from_tst, date_to_tst))
        .filter(F.col("event_id").isin(event_ids))
        .selectExpr("event_id", "cast(atm_mcc as string) as mcc_code")
    )
    return atm_merchant.toPandas()


def load_mcc_dictionary() -> pd.DataFrame:
    """Загружает справочник MCC из PostgreSQL и оставляет поля для обогащения кейсов.

    Args:
        Нет входных аргументов. Используется внешний объект `eng_pg`.

    Returns:
        pandas DataFrame с колонками `mcc_code`, `mcc_name`, `description`.
    """

    mcc_request = """
    SELECT a.*, b.*
    FROM selfdags.mcc_codes a
    LEFT JOIN dashboards.mcc_dictionary b
    ON CAST(a.mcc_code AS INT) = CAST(b.mcc AS INT)
    """
    mcc_pd = pd.read_sql(mcc_request, eng_pg).drop(columns=["mcc", "activity"], errors="ignore")
    mcc_pd = mcc_pd[["mcc_code", "mcc_name", "description"]].drop_duplicates()
    mcc_pd["mcc_code"] = mcc_pd["mcc_code"].astype(str)
    return mcc_pd


def enrich_mcc_names(df: pd.DataFrame, mcc_dictionary: pd.DataFrame) -> pd.DataFrame:
    """Добавляет описание MCC и применяет ручные бизнес-переопределения названий.

    Args:
        df: pandas DataFrame с колонкой `mcc_code`.
        mcc_dictionary: Справочник MCC с колонками `mcc_code`, `mcc_name`, `description`.

    Returns:
        pandas DataFrame с добавленными колонками `mcc_name` и `description`.
    """

    result = df.copy()
    result["mcc_code"] = result["mcc_code"].astype(str)
    result = result.merge(mcc_dictionary, on="mcc_code", how="left")
    for mcc_code, mcc_name in MCC_NAME_OVERRIDES.items():
        result["mcc_name"] = np.where(result["mcc_code"] == mcc_code, mcc_name, result["mcc_name"])
    return result

## 4. Загрузка событий клиента за день сработки

In [ ]:
def empty_events_schema() -> T.StructType:
    """Создает схему пустого Spark DataFrame для событий клиента.

    Args:
        Нет входных аргументов.

    Returns:
        Spark StructType со всеми колонками, нужными для unionByName.
    """

    base_fields = [
        T.StructField("epk_id", T.LongType(), True),
        T.StructField("event_dt", T.StringType(), True),
        T.StructField("event_type", T.StringType(), True),
        T.StructField("sub_type", T.StringType(), True),
        T.StructField("event_channel", T.StringType(), True),
        T.StructField("event_description", T.StringType(), True),
        T.StructField("transaction_amount", T.StringType(), True),
        T.StructField("atm_mcc", T.StringType(), True),
        T.StructField("evt_time", T.TimestampType(), True),
    ]
    return T.StructType(base_fields + [T.StructField(column, T.StringType(), True) for column in EVENT_EXTRA_COLUMNS])


def load_events_for_source(table_name: str, keys: SparkDataFrame, has_extra: bool) -> SparkDataFrame:
    """Загружает события за день сработки по точным парам `epk_id` и `event_dt`.

    Args:
        table_name: Название Spark-таблицы с событиями.
        keys: Spark DataFrame с уникальными парами `epk_id`, `event_dt`.
        has_extra: Признак наличия дополнительных UKO-полей в таблице.

    Returns:
        Spark DataFrame с унифицированной схемой событий.
    """

    if keys.rdd.isEmpty():
        return spark.createDataFrame([], schema=empty_events_schema())

    source = (
        spark.read.table(table_name)
        .withColumn("epk_id", F.col("epk_id").cast("long"))
        .withColumn("event_dt", F.col("event_dt").cast("string"))
    )
    joined = source.join(keys, on=["epk_id", "event_dt"], how="inner")

    select_list = [
        F.col("epk_id").cast("long").alias("epk_id"),
        F.col("event_dt").cast("string").alias("event_dt"),
        F.col("event_type").cast("string").alias("event_type"),
        F.col("sub_type").cast("string").alias("sub_type"),
        F.col("event_channel").cast("string").alias("event_channel"),
        F.col("event_description").cast("string").alias("event_description"),
        F.col("transaction_amount").cast("string").alias("transaction_amount"),
        F.col("atm_mcc").cast("string").alias("atm_mcc"),
        F.from_unixtime(F.col("event_time").cast("long") / 1000).cast("timestamp").alias("evt_time"),
    ]

    if has_extra:
        select_list.extend([F.col(column).cast("string").alias(column) for column in EVENT_EXTRA_COLUMNS])
    else:
        select_list.extend([F.lit(None).cast("string").alias(column) for column in EVENT_EXTRA_COLUMNS])

    return joined.select(*select_list)


def load_day_events(config: ExportConfig, cases_pdf: pd.DataFrame) -> pd.DataFrame:
    """Загружает все события клиента за день сработки и присоединяет их к базовым кейсам.

    Args:
        config: Конфигурация выгрузки.
        cases_pdf: pandas DataFrame с базовыми кейсами, включая `epk_id`, `event_dt`, `channel`.

    Returns:
        pandas DataFrame с колонкой `events_day`, содержащей отсортированный список событий за день.
    """

    hits = safe_to_spark(cases_pdf)
    base_keys = hits.select(F.col("epk_id").cast("long").alias("epk_id"), F.col("event_dt").cast("string").alias("event_dt"), "channel")

    cards_keys = base_keys.filter(F.col("channel") == "Эмиссия").select("epk_id", "event_dt").distinct()
    uko_keys = base_keys.filter((F.col("channel") != "Эмиссия") | F.col("channel").isNull()).select("epk_id", "event_dt").distinct()

    events_cards = load_events_for_source(config.cards_event_table, cards_keys, has_extra=False)
    events_uko = load_events_for_source(config.uko_event_table, uko_keys, has_extra=True)
    all_events = events_cards.unionByName(events_uko)

    events_col = (
        all_events.groupBy("epk_id", "event_dt")
        .agg(F.sort_array(F.collect_list(F.struct(*EVENT_STRUCT_COLUMNS))).alias("events_day"))
    )

    result = hits.join(events_col, on=["epk_id", "event_dt"], how="left")
    for column in ["event_time", "answer_date", "answer_time", "first_resolution_time", "last_resolution_time"]:
        if column in result.columns:
            result = result.withColumn(column, F.date_format(F.col(column), "yyyy-MM-dd HH:mm:ss"))
    return result.toPandas()

## 5. Сборка базового dataset

In [ ]:
def build_base_case_dataset(config: ExportConfig) -> pd.DataFrame:
    """Собирает базовый dataset с CSI, сработками, MCC и событиями клиента за день.

    Args:
        config: Конфигурация выгрузки.

    Returns:
        pandas DataFrame, готовый для форматирования в строковые запросы агенту.
    """

    _, history_date_from_local, date_from_tst_local, date_to_tst_local = build_week_windows(config)

    csi_spark = load_csi_answers(config, history_date_from_local)
    csi_pdf = csi_spark.toPandas()
    cases_pdf = transform_survey_data_clean(csi_pdf).drop(columns=["answer_8"], errors="ignore")
    cases_pdf["event_id"] = cases_pdf["event_id"].astype(str)

    event_ids = cases_pdf["event_id"].dropna().astype(str).unique().tolist()
    hits_extra_pdf = load_hits_extra(config, event_ids, date_from_tst_local, date_to_tst_local)
    hits_extra_pdf["event_id"] = hits_extra_pdf["event_id"].astype(str)

    cases_pdf = cases_pdf.merge(hits_extra_pdf, on="event_id", how="left")
    cases_pdf = coalesce_duplicate_columns(cases_pdf)
    cases_pdf = add_business_channel(cases_pdf)

    atm_mcc_pdf = load_atm_mcc(config, event_ids, date_from_tst_local, date_to_tst_local)
    atm_mcc_pdf["event_id"] = atm_mcc_pdf["event_id"].astype(str)
    cases_pdf = cases_pdf.merge(atm_mcc_pdf, on="event_id", how="left")

    mcc_dictionary = load_mcc_dictionary()
    cases_pdf = enrich_mcc_names(cases_pdf, mcc_dictionary)

    result_pdf = load_day_events(config, cases_pdf)
    return result_pdf


# Запуск полной выгрузки. Ячейка не выполняется автоматически при открытии notebook.
result_pandas = build_base_case_dataset(CONFIG)
print(f"Готово: {result_pandas.shape[0]} строк, {result_pandas.shape[1]} колонок.")
result_pandas.head(3)

## 6. Форматирование строк для передачи агенту

`agent_requests` можно передавать агенту как обычный текстовый запрос. В начале добавляется инструкция, а ниже идет полный контекст одной строки.

In [ ]:
def parse_complex_string(value: Any) -> Any:
    """Пробует разобрать строку, если она похожа на JSON, list или dict.

    Args:
        value: Произвольное значение из DataFrame.

    Returns:
        Исходное значение или разобранный Python-объект.
    """

    if not isinstance(value, str):
        return value
    stripped = value.strip()
    if (stripped.startswith("{") and stripped.endswith("}")) or (stripped.startswith("[") and stripped.endswith("]")):
        try:
            return json.loads(stripped)
        except Exception:
            try:
                return ast.literal_eval(stripped)
            except Exception:
                return value
    return value


def format_complex_value(value: Any, level: int = 0, max_string_length: int | None = None) -> str:
    """Форматирует сложное значение в читаемый многострочный текст.

    Args:
        value: Значение ячейки DataFrame.
        level: Уровень вложенности для отступов.
        max_string_length: Максимальная длина строки. Если `None`, строка не обрезается.

    Returns:
        Строковое представление значения для prompt-а агента.
    """

    indent = "  " * level
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return f"{indent}Значение отсутствует"
    if isinstance(value, (datetime, pd.Timestamp)):
        return f"{indent}{value.strftime('%Y-%m-%d %H:%M:%S')}"
    if isinstance(value, dict):
        if not value:
            return f"{indent}{{}}"
        lines = [f"{indent}{{"]
        for key, nested_value in value.items():
            lines.append(f"{indent}  {key}: {format_complex_value(nested_value, level + 1, max_string_length).lstrip()}")
        lines.append(f"{indent}}}")
        return "\n".join(lines)
    if isinstance(value, (list, tuple, np.ndarray)):
        if len(value) == 0:
            return f"{indent}[]"
        lines = [f"{indent}["]
        for item in value:
            formatted_item = format_complex_value(item, level + 1, max_string_length)
            lines.append(f"{indent}  {formatted_item.lstrip()}")
        lines.append(f"{indent}]")
        return "\n".join(lines)
    if isinstance(value, str):
        parsed = parse_complex_string(value)
        if parsed != value:
            return format_complex_value(parsed, level, max_string_length)
        text = value if max_string_length is None or len(value) <= max_string_length else f"{value[:max_string_length]}..."
        return f"{indent}\"{text}\""
    return f"{indent}{str(value)}"


def format_dataframe_rows_to_list(df: pd.DataFrame, max_string_length: int | None = None) -> list[str]:
    """Преобразует каждую строку DataFrame в отдельный читаемый текстовый блок.

    Args:
        df: pandas DataFrame с базовым контекстом кейсов.
        max_string_length: Максимальная длина строковых значений. Если `None`, значения не обрезаются.

    Returns:
        Список строк, где каждый элемент является полным контекстом одного кейса.
    """

    result_rows: list[str] = []
    for row_number, (_, row) in enumerate(df.iterrows(), start=1):
        row_lines = ["=" * 100, f"Строка №{row_number}", "=" * 100]
        for column in df.columns:
            row_lines.append(f"\n{column}:")
            formatted = format_complex_value(row[column], max_string_length=max_string_length)
            row_lines.extend(f"  {line}" for line in formatted.split("\n"))
        row_lines.append("=" * 100)
        result_rows.append("\n".join(row_lines))
    return result_rows


def build_agent_requests(df: pd.DataFrame, max_string_length: int | None = None) -> list[str]:
    """Формирует обычные текстовые запросы агенту по каждой строке dataset.

    Args:
        df: pandas DataFrame с базовым контекстом кейсов.
        max_string_length: Максимальная длина строковых значений в контексте.

    Returns:
        Список готовых prompt-строк для запуска агента по одному кейсу.
    """

    instruction = """
Разбери антифрод-сработку по данным ниже.

Требования:
1. Определи, достаточно ли текущего базового контекста для объяснения кейса.
2. Если данных достаточно, опиши, что произошло, какие факты это подтверждают и какие есть ограничения.
3. Если данных не хватает, перечисли, какие данные нужно дополнительно загрузить, по каким ключам и зачем.
4. Не выдумывай факты. Отделяй подтвержденные факты от гипотез.
5. В конце верни структурированный блок: факты, гипотезы, недостающие данные, ограничения, итог.
""".strip()

    return [f"{instruction}\n\n{row_text}" for row_text in format_dataframe_rows_to_list(df, max_string_length=max_string_length)]


agent_requests = build_agent_requests(result_pandas, max_string_length=None)
print(f"Готово запросов для агента: {len(agent_requests)}")
print(agent_requests[0][:4000] if agent_requests else "Нет строк для передачи агенту.")

## 7. Как расширять выгрузку

1. Новые поля из `hits_extra` добавляй в `HITS_EXTRA_COLUMNS`.
2. Новые поля из `uko_event` добавляй в `EVENT_EXTRA_COLUMNS` и при необходимости в `EVENT_STRUCT_COLUMNS`.
3. Новый справочник лучше оформлять отдельной функцией `load_*_dictionary()` и отдельной функцией `enrich_*()`.
4. Если источник нужен не всем кейсам, добавляй его после базовой сборки отдельным left join по ключам `event_id`, `epk_id`, `event_dt` или другой явной связке.
5. Для больших списков `event_id` лучше заменить `.isin(event_ids)` на join с временным Spark DataFrame ключей.